In [1]:
import torch
import sentence_transformers
import rank_bm25

print("PyTorch:", torch.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
Sentence Transformers: 6.1.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

print("Embedding model loaded")
print("Device:", device)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Projects\DocuMind\ml\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sudee\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Device: cuda
Embedding dimension: 384


C:\Users\sudee\AppData\Local\Temp\ipykernel_41140\4120068616.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [3]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../processed")

train_df = pd.read_csv(processed_dir / "train.csv")
val_df = pd.read_csv(processed_dir / "validation.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

search_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("Documents:", len(search_df))
print("Columns:", search_df.columns.tolist())
print("\nClass distribution:")
print(search_df["label"].value_counts())
print("\nDuplicate document IDs:", search_df["document_id"].duplicated().sum())

Documents: 6638
Columns: ['document_id', 'model_text', 'label', 'source']

Class distribution:
label
Report            2000
Email             2000
Invoice           2000
Contract           510
Purchase Order     128
Name: count, dtype: int64

Duplicate document IDs: 0


In [4]:
def chunk_text(text, chunk_size=350, overlap=75):
    words = str(text).split()

    if len(words) <= chunk_size:
        return [str(text).strip()]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunk_rows = []

for _, row in search_df.iterrows():

    chunks = chunk_text(
        row["model_text"],
        chunk_size=350,
        overlap=75
    )

    for chunk_idx, chunk in enumerate(chunks):

        chunk_rows.append({
            "chunk_id": f"{row['document_id']}_{chunk_idx}",
            "document_id": row["document_id"],
            "label": row["label"],
            "source": row["source"],
            "chunk_index": chunk_idx,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunk_rows)

print("Documents:", chunks_df["document_id"].nunique())
print("Total chunks:", len(chunks_df))
print("\nChunks per document:")
print(chunks_df.groupby("document_id").size().describe())

Documents: 6638
Total chunks: 16851

Chunks per document:
count    6638.000000
mean        2.538566
std         1.771811
min         1.000000
25%         1.000000
50%         1.000000
75%         5.000000
max         6.000000
dtype: float64


In [5]:
import numpy as np

texts = chunks_df["text"].tolist()

print("Generating embeddings for", len(texts), "chunks...")

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nEmbedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Generating embeddings for 16851 chunks...


Batches:   0%|          | 0/527 [00:00<?, ?it/s]


Embedding shape: (16851, 384)
Embedding dtype: float32


In [6]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    text.lower().split()
    for text in chunks_df["text"]
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready")
print("Indexed chunks:", len(tokenized_corpus))

BM25 index ready
Indexed chunks: 16851


In [7]:
def min_max_normalize(scores):
    scores = np.asarray(scores, dtype=np.float32)

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.ones_like(scores)

    return (scores - min_score) / (max_score - min_score)


def hybrid_search(
    query,
    top_k=5,
    candidate_k=20,
    alpha=0.5
):
    """
    Hybrid retrieval using:
        alpha * BM25
        + (1-alpha) * semantic similarity
    """

    # -----------------------------------------------------
    # 1. BM25 retrieval
    # -----------------------------------------------------
    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    bm25_top_indices = np.argsort(
        bm25_scores
    )[::-1][:candidate_k]


    # -----------------------------------------------------
    # 2. Semantic retrieval
    # -----------------------------------------------------
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = embeddings @ query_embedding

    semantic_top_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]


    # -----------------------------------------------------
    # 3. Combine candidate documents
    # -----------------------------------------------------
    candidate_indices = sorted(
        set(bm25_top_indices.tolist()) |
        set(semantic_top_indices.tolist())
    )

    candidate_bm25 = np.array([
        bm25_scores[i]
        for i in candidate_indices
    ])

    candidate_semantic = np.array([
        semantic_scores[i]
        for i in candidate_indices
    ])


    # -----------------------------------------------------
    # 4. Normalize scores
    # -----------------------------------------------------
    bm25_normalized = min_max_normalize(
        candidate_bm25
    )

    semantic_normalized = min_max_normalize(
        candidate_semantic
    )


    # -----------------------------------------------------
    # 5. Weighted fusion
    # -----------------------------------------------------
    hybrid_scores = (
        alpha * bm25_normalized
        + (1 - alpha) * semantic_normalized
    )


    # -----------------------------------------------------
    # 6. Rank results
    # -----------------------------------------------------
    ranked_order = np.argsort(
        hybrid_scores
    )[::-1][:top_k]


    result_indices = [
        candidate_indices[i]
        for i in ranked_order
    ]


    results = chunks_df.iloc[
        result_indices
    ].copy()

    results["bm25_score"] = [
        candidate_bm25[i]
        for i in ranked_order
    ]

    results["semantic_score"] = [
        candidate_semantic[i]
        for i in ranked_order
    ]

    results["hybrid_score"] = [
        hybrid_scores[i]
        for i in ranked_order
    ]

    return results.reset_index(drop=True)

In [8]:
results = hybrid_search(
    "contracts with early termination penalties",
    top_k=5
)

display(
    results[
        [
            "document_id",
            "label",
            "hybrid_score",
            "text"
        ]
    ]
)

,document_id,label,hybrid_score,text
0,contract_0356,Contract,0.873980,of the Agreement by both parties and shall ter...
1,contract_0437,Contract,0.861580,Obligations 17 18.2. Exceptions to Obligations...
2,contract_0159,Contract,0.813834,"in accordance with this Agreement. However, Pa..."
3,email_1873,Email,0.786905,"John, Managing VAR limits is essential and P&L..."
4,contract_0441,Contract,0.717328,(30) days from receipt of notice to cure such ...


In [9]:
test_queries = [
    "contracts with early termination penalties",
    "invoice payment due date and total amount",
    "purchase order delivery date and supplier",
    "emails discussing financial results",
    "agreement can be terminated before expiration"
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = hybrid_search(
        query,
        top_k=3
    )

    display(
        results[
            [
                "document_id",
                "label",
                "hybrid_score",
                "text"
            ]
        ]
    )


QUERY: contracts with early termination penalties


,document_id,label,hybrid_score,text
0,contract_0356,Contract,0.873980,of the Agreement by both parties and shall ter...
1,contract_0437,Contract,0.861580,Obligations 17 18.2. Exceptions to Obligations...
2,contract_0159,Contract,0.813834,"in accordance with this Agreement. However, Pa..."



QUERY: invoice payment due date and total amount


,document_id,label,hybrid_score,text
0,invoice_0113,Invoice,0.844368,EArns: INVOICE DATE CUSTOMER NUN 5/26/99 PLEAS...
1,invoice_0474,Invoice,0.801320,TOTAL THAT THE ACTUAL SHOWN: ON THIS INVOICE W...
2,invoice_1224,Invoice,0.771373,DAY TIME LENGIH MGFOR PRODVET DESCRPTION E DUP...



QUERY: purchase order delivery date and supplier


,document_id,label,hybrid_score,text
0,contract_0107,Contract,1.000000,from Supplier by issuing a purchase order that...
1,contract_0035,Contract,0.784467,email address set out in Schedule 1. Each Orde...
2,contract_0035,Contract,0.619834,reasonable detail (and unless and solely to th...



QUERY: emails discussing financial results


,document_id,label,hybrid_score,text
0,email_0825,Email,0.795499,All-Employee Meeting I want to remind you abou...
1,email_0906,Email,0.748590,TheStreet.com: http://www.thestreet.com/tilex/...
2,report_0629,Report,0.742107,BNY MELLON 2017 ANNUAL REPORT ## We are just b...



QUERY: agreement can be terminated before expiration


,document_id,label,hybrid_score,text
0,contract_0365,Contract,0.946700,be made in installments according to the sched...
1,contract_0032,Contract,0.856894,"date of the operation term of Party B (""Term o..."
2,contract_0159,Contract,0.819878,"in accordance with this Agreement. However, Pa..."


In [10]:
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Evaluation queries
# Each query has an expected document class.
# ---------------------------------------------------------

eval_queries = [
    # Contracts
    ("early termination penalties in an agreement", "Contract"),
    ("contract renewal and termination conditions", "Contract"),
    ("liability obligations between the parties", "Contract"),
    ("agreement expiration and notice requirements", "Contract"),
    ("confidentiality obligations in a contract", "Contract"),

    # Invoices
    ("invoice total amount and payment due date", "Invoice"),
    ("billing invoice with customer and vendor", "Invoice"),
    ("invoice outstanding balance", "Invoice"),
    ("invoice payment terms and amount due", "Invoice"),
    ("total payable on the invoice", "Invoice"),

    # Purchase Orders
    ("purchase order supplier and delivery date", "Purchase Order"),
    ("purchase order number and vendor", "Purchase Order"),
    ("items ordered from supplier", "Purchase Order"),
    ("purchase order shipping information", "Purchase Order"),
    ("supplier purchase order details", "Purchase Order"),

    # Emails
    ("employees discussing financial results", "Email"),
    ("internal email about business performance", "Email"),
    ("email discussing company earnings", "Email"),
    ("email communication with financial updates", "Email"),
    ("internal discussion about revenue", "Email"),

    # Reports
    ("annual report financial performance", "Report"),
    ("company yearly financial results", "Report"),
    ("annual report revenue and earnings", "Report"),
    ("financial statements in an annual report", "Report"),
    ("company performance report", "Report"),
]


# ---------------------------------------------------------
# Generic ranking functions
# ---------------------------------------------------------

def bm25_search(query, top_k=5):
    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    indices = np.argsort(scores)[::-1][:top_k]

    return chunks_df.iloc[indices].copy()


def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    scores = embeddings @ query_embedding

    indices = np.argsort(scores)[::-1][:top_k]

    return chunks_df.iloc[indices].copy()


# ---------------------------------------------------------
# Evaluate Recall@K
# ---------------------------------------------------------

evaluation_rows = []

for query, expected_label in eval_queries:

    bm25_results = bm25_search(
        query,
        top_k=5
    )

    semantic_results = semantic_search(
        query,
        top_k=5
    )

    hybrid_results = hybrid_search(
        query,
        top_k=5
    )

    bm25_hit = (
        expected_label
        in bm25_results["label"].values
    )

    semantic_hit = (
        expected_label
        in semantic_results["label"].values
    )

    hybrid_hit = (
        expected_label
        in hybrid_results["label"].values
    )

    evaluation_rows.append({
        "query": query,
        "expected_label": expected_label,
        "bm25_recall@5": bm25_hit,
        "semantic_recall@5": semantic_hit,
        "hybrid_recall@5": hybrid_hit
    })


retrieval_eval_df = pd.DataFrame(
    evaluation_rows
)


print("Queries:", len(retrieval_eval_df))

print("\nRecall@5:")
print(
    retrieval_eval_df[
        [
            "bm25_recall@5",
            "semantic_recall@5",
            "hybrid_recall@5"
        ]
    ].mean()
)

Queries: 25

Recall@5:
bm25_recall@5        0.88
semantic_recall@5    0.84
hybrid_recall@5      0.84
dtype: float64


In [11]:
def hybrid_search_rrf(query, top_k=5, candidate_k=20, k=60):
    query_tokens = query.lower().split()

    # BM25 ranking
    bm25_scores = bm25.get_scores(query_tokens)

    bm25_ranked = np.argsort(
        bm25_scores
    )[::-1][:candidate_k]

    # Semantic ranking
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = embeddings @ query_embedding

    semantic_ranked = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # RRF scores
    rrf_scores = {}

    for rank, idx in enumerate(bm25_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    for rank, idx in enumerate(semantic_ranked):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)


print("RRF hybrid search ready")

RRF hybrid search ready


In [12]:
results = hybrid_search_rrf(
    "contracts with early termination penalties",
    top_k=5
)

display(
    results[
        ["document_id", "label", "rrf_score", "text"]
    ]
)

,document_id,label,rrf_score,text
0,contract_0356,Contract,0.031778,of the Agreement by both parties and shall ter...
1,contract_0437,Contract,0.029828,Obligations 17 18.2. Exceptions to Obligations...
2,contract_0159,Contract,0.029031,"in accordance with this Agreement. However, Pa..."
3,contract_0441,Contract,0.028543,(30) days from receipt of notice to cure such ...
4,contract_0320,Contract,0.026334,"by law, government regulation, or court order...."


In [13]:
# Evaluate RRF hybrid search on the same 25 queries

rrf_hits = []

for query, expected_label in eval_queries:

    results = hybrid_search_rrf(
        query,
        top_k=5
    )

    hit = (
        expected_label
        in results["label"].values
    )

    rrf_hits.append(hit)


rrf_recall_at_5 = np.mean(rrf_hits)

print("Queries:", len(eval_queries))
print("RRF Hybrid Recall@5:", round(rrf_recall_at_5, 4))

Queries: 25
RRF Hybrid Recall@5: 0.88


In [14]:
from pathlib import Path
import pickle

search_artifacts = Path("../models/hybrid_search")
search_artifacts.mkdir(parents=True, exist_ok=True)

# Save chunks
chunks_df.to_parquet(
    search_artifacts / "chunks.parquet",
    index=False
)

# Save embeddings
np.save(
    search_artifacts / "embeddings.npy",
    embeddings
)

# Save BM25 index
with open(
    search_artifacts / "bm25.pkl",
    "wb"
) as f:
    pickle.dump(bm25, f)

print("Search artifacts saved.")
print("Location:", search_artifacts)

Search artifacts saved.
Location: ..\models\hybrid_search


In [15]:
print(search_df.columns.tolist())

['document_id', 'model_text', 'label', 'source']


In [1]:
%pip install bm25s

Note: you may need to restart the kernel to use updated packages.


In [2]:
import bm25s

print("bm25s version:", bm25s.__version__)
print("BM25S import successful")

bm25s version: 0.3.11
BM25S import successful


In [3]:
# filepath: ml/notebooks/04_hybrid_search.ipynb

import bm25s
from pathlib import Path
import pyarrow.parquet as pq

processed_dir = Path("../processed")

chunks_path = processed_dir / "full_text_chunks.parquet"

parquet_file = pq.ParquetFile(chunks_path)

print("Chunk rows:", parquet_file.metadata.num_rows)

print(
    "BM25S version:",
    bm25s.__version__
)

print("BM25S ready.")

Chunk rows: 223234
BM25S version: 0.3.11
BM25S ready.


In [4]:
import time
import pyarrow.parquet as pq
import bm25s

chunks_path = "../processed/full_text_chunks.parquet"

parquet_file = pq.ParquetFile(chunks_path)

# Load only the first 10,000 chunk texts
pilot_texts = []

for batch in parquet_file.iter_batches(
    batch_size=10000,
    columns=["text"]
):
    pilot_texts.extend(
        batch.to_pandas()["text"].tolist()
    )

    if len(pilot_texts) >= 10000:
        break

pilot_texts = pilot_texts[:10000]

print("Pilot chunks:", len(pilot_texts))

# Tokenize
start = time.time()

pilot_tokens = bm25s.tokenize(
    pilot_texts,
    stopwords="en",
    show_progress=False
)

print(
    "Tokenization time:",
    round(time.time() - start, 2),
    "seconds"
)

# Build index
start = time.time()

pilot_bm25 = bm25s.BM25()
pilot_bm25.index(
    pilot_tokens,
    show_progress=False
)

print(
    "Indexing time:",
    round(time.time() - start, 2),
    "seconds"
)

print("BM25S pilot ready.")

Pilot chunks: 10000
Tokenization time: 1.93 seconds
Indexing time: 0.92 seconds
BM25S pilot ready.


In [5]:
import bm25s
import pyarrow.parquet as pq
from pathlib import Path
import time

chunks_path = Path("../processed/full_text_chunks.parquet")
bm25s_dir = Path("../processed/full_text_bm25s")
bm25s_dir.mkdir(parents=True, exist_ok=True)

# Load all chunk text
print("Loading chunk text...")

parquet_file = pq.ParquetFile(
    chunks_path
)

all_texts = []

for batch in parquet_file.iter_batches(
    batch_size=10000,
    columns=["text"]
):
    all_texts.extend(
        batch.to_pandas()["text"].tolist()
    )

print("Chunks loaded:", len(all_texts))

# ---------------------------------------------------------
# Tokenization
# ---------------------------------------------------------
start = time.time()

tokens = bm25s.tokenize(
    all_texts,
    stopwords="en",
    show_progress=True
)

print(
    "Tokenization time:",
    round(time.time() - start, 2),
    "seconds"
)

# ---------------------------------------------------------
# Build BM25S index
# ---------------------------------------------------------
start = time.time()

full_bm25s = bm25s.BM25()

full_bm25s.index(
    tokens,
    show_progress=True
)

print(
    "Indexing time:",
    round(time.time() - start, 2),
    "seconds"
)

# ---------------------------------------------------------
# Save index
# ---------------------------------------------------------
full_bm25s.save(
    str(bm25s_dir),
    corpus=None
)

print("\nBM25S full index saved.")
print("Indexed chunks:", len(all_texts))
print("Index directory:", bm25s_dir)

Loading chunk text...
Chunks loaded: 223234


Split strings:   0%|          | 0/223234 [00:00<?, ?it/s]

Tokenization time: 101.21 seconds


BM25S Count Tokens:   0%|          | 0/223234 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/223234 [00:00<?, ?it/s]

Indexing time: 39.17 seconds

BM25S full index saved.
Indexed chunks: 223234
Index directory: ..\processed\full_text_bm25s


In [6]:
import numpy as np
import pandas as pd
import bm25s
from pathlib import Path

processed_dir = Path("../processed")

# Load chunk metadata + text
full_chunks_df = pd.read_parquet(
    processed_dir / "full_text_chunks.parquet"
)

# Load embeddings without copying the whole array into RAM
full_embeddings = np.load(
    processed_dir / "full_text_embeddings.npy",
    mmap_mode="r"
)

# Load BM25S index
full_bm25s = bm25s.BM25.load(
    str(processed_dir / "full_text_bm25s"),
    load_corpus=False
)

print("Final search artifacts loaded.")
print("Chunks:", len(full_chunks_df))
print("Embeddings:", full_embeddings.shape)
print("BM25S ready.")

Final search artifacts loaded.
Chunks: 223234
Embeddings: (223234, 384)
BM25S ready.


In [12]:
import numpy as np


def hybrid_search_full_text(
    query,
    top_k=5,
    candidate_k=30,
    rrf_k=60,
    label_filter=None
):
    # -----------------------------------------------------
    # Optional document-type filter
    # -----------------------------------------------------
    if label_filter is not None:
        allowed_mask = (
            full_chunks_df["label"].values == label_filter
        )
    else:
        allowed_mask = np.ones(
            len(full_chunks_df),
            dtype=bool
        )

    # -----------------------------------------------------
    # 1. BM25S search
    # -----------------------------------------------------
    query_tokens = bm25s.tokenize(
        [query],
        stopwords="en"
    )

    bm25_results, bm25_scores = full_bm25s.retrieve(
        query_tokens,
        k=candidate_k,
        show_progress=False
    )

    # Convert to 1D arrays
    bm25_indices = np.asarray(
        bm25_results[0]
    ).astype(int)

    bm25_scores = np.asarray(
        bm25_scores[0]
    )

    # Remove filtered results
    filtered_bm25 = []

    for idx in bm25_indices:
        if allowed_mask[idx]:
            filtered_bm25.append(idx)

    bm25_indices = np.array(
        filtered_bm25[:candidate_k],
        dtype=int
    )

    # -----------------------------------------------------
    # 2. Semantic BGE search
    # -----------------------------------------------------
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = full_embeddings @ query_embedding

    semantic_scores = np.asarray(
        semantic_scores,
        dtype=np.float32
    )

    semantic_scores[~allowed_mask] = -np.inf

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # -----------------------------------------------------
    # 3. Reciprocal Rank Fusion
    # -----------------------------------------------------
    rrf_scores = {}

    for rank, idx in enumerate(bm25_indices):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    for rank, idx in enumerate(semantic_indices):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    ranked_indices = sorted(
        rrf_scores.keys(),
        key=lambda idx: rrf_scores[idx],
        reverse=True
    )[:top_k]

    # -----------------------------------------------------
    # 4. Build result dataframe
    # -----------------------------------------------------
    results = full_chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    results["semantic_score"] = [
        semantic_scores[idx]
        for idx in ranked_indices
    ]

    results["bm25_ranked"] = [
        idx in bm25_indices
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)


print("Full-text hybrid search ready")

Full-text hybrid search ready


In [13]:
import numpy as np


def hybrid_search_full_text(
    query,
    top_k=5,
    candidate_k=30,
    rrf_k=60,
    label_filter=None
):
    # -----------------------------------------------------
    # Optional document-type filter
    # -----------------------------------------------------
    if label_filter is not None:
        allowed_mask = (
            full_chunks_df["label"].values == label_filter
        )
    else:
        allowed_mask = np.ones(
            len(full_chunks_df),
            dtype=bool
        )

    # -----------------------------------------------------
    # 1. BM25S search
    # -----------------------------------------------------
    query_tokens = bm25s.tokenize(
        [query],
        stopwords="en"
    )

    bm25_results, bm25_scores = full_bm25s.retrieve(
        query_tokens,
        k=candidate_k,
        show_progress=False
    )

    # Convert to 1D arrays
    bm25_indices = np.asarray(
        bm25_results[0]
    ).astype(int)

    bm25_scores = np.asarray(
        bm25_scores[0]
    )

    # Remove filtered results
    filtered_bm25 = []

    for idx in bm25_indices:
        if allowed_mask[idx]:
            filtered_bm25.append(idx)

    bm25_indices = np.array(
        filtered_bm25[:candidate_k],
        dtype=int
    )

    # -----------------------------------------------------
    # 2. Semantic BGE search
    # -----------------------------------------------------
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = full_embeddings @ query_embedding

    semantic_scores = np.asarray(
        semantic_scores,
        dtype=np.float32
    )

    semantic_scores[~allowed_mask] = -np.inf

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # -----------------------------------------------------
    # 3. Reciprocal Rank Fusion
    # -----------------------------------------------------
    rrf_scores = {}

    for rank, idx in enumerate(bm25_indices):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    for rank, idx in enumerate(semantic_indices):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    ranked_indices = sorted(
        rrf_scores.keys(),
        key=lambda idx: rrf_scores[idx],
        reverse=True
    )[:top_k]

    # -----------------------------------------------------
    # 4. Build result dataframe
    # -----------------------------------------------------
    results = full_chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    results["semantic_score"] = [
        semantic_scores[idx]
        for idx in ranked_indices
    ]

    results["bm25_ranked"] = [
        idx in bm25_indices
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)


print("Full-text hybrid search ready")

Full-text hybrid search ready


In [9]:
print(
    "Embedding model:",
    type(embedding_model).__name__
)

print(
    "Embedding dimension:",
    embedding_model.get_embedding_dimension()
)

NameError: name 'embedding_model' is not defined

In [10]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

print("Embedding model loaded")
print("Device:", device)
print("Embedding dimension:", embedding_model.get_embedding_dimension())

NameError: name 'torch' is not defined

In [11]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

print("Embedding model loaded")
print("Device:", device)
print(
    "Embedding dimension:",
    embedding_model.get_embedding_dimension()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded
Device: cuda
Embedding dimension: 384


In [14]:
query = "Which contracts mention early termination penalties?"

results = hybrid_search_full_text(
    query,
    top_k=5,
    candidate_k=30,
    label_filter="Contract"
)

display(
    results[
        [
            "document_id",
            "label",
            "chunk_index",
            "rrf_score",
            "text"
        ]
    ]
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

,document_id,label,chunk_index,rrf_score,text
0,contract_0112,Contract,2,0.016393,notice if the other party: (w) becomes insolve...
1,contract_0086,Contract,16,0.016393,of this Agreement before any court having juri...
2,contract_0356,Contract,1,0.016129,for work performed to date of receipt of termi...
3,contract_0229,Contract,20,0.015873,may directly or indirectly incur by reason of ...
4,contract_0154,Contract,6,0.015625,to any other remedies available to either Part...


In [15]:
for i, row in results.iterrows():
    print("\n" + "=" * 100)
    print(
        f"Rank: {i + 1}\n"
        f"Document: {row['document_id']}\n"
        f"Chunk: {row['chunk_index']}\n"
        f"RRF score: {row['rrf_score']:.6f}\n"
    )
    print(row["text"])


Rank: 1
Document: contract_0112
Chunk: 2
RRF score: 0.016393

notice if the other party: (w) becomes insolvent; (x) files a petition in bankruptcy; (y) makes an assignment for the benefit of its creditors; or (z) breach any of its obligations under this Agreement in any material respect, which breach is not remedied within thirty (30) days following written notice to such party. EFFECT OF TERMINATION: Any termination shall be without any liability or obligation of the terminating party, other than with respect to any breach of this Agreement prior to termination. The provisions relating to property rights and confidentiality shall survive any termination or expiration of this Agreement. All revenue sharing ceases with the termination of this Agreement. Initialed THE HENRY FILM AND ENTERTAINMENT CORPORATION:______ Initialed PACIFICAP ENTERTAINMENT:______ Page 3 of 6 Source: PACIFICAP ENTERTAINMENT HOLDINGS INC, 8-K/A, 11/15/2005 PACIFICAP ENTERTAINMENT Agreement with THE HENRY FILM AND